In [17]:
import pyexasol
import configparser
import pandas as pd
from datetime import date
import re

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\pfxPROD.ini')

dsn=config['pfxPROD']['dsn']
user=config['pfxPROD']['user']
pwd=config['pfxPROD']['pwd']
schema=config['pfxPROD']['schema']
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)

In [33]:
### LOCATION OF THE FILE
path = 'C:\\Users\\USER\\Documents\\HPO\HS_Files\\Archiv\\'
# filename = 'GREYLIST_(2018-07-16).xlsx'
filename = 'BLACKLIST_(2019-09-27).xlsx'
string = re.split('_', filename)[0]
df = pd.read_excel (path+filename)
df.rename(columns={'Blacklist or Graylist': 'FILTER_STATUS', 'Hkey' : 'HOTEL_ID'}, errors='raise', inplace=True)
df['FILTER_LOAD_DATE'] = date.today()

df = df[df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()]
df['FILTER_STATUS'] = string

connect.execute(f"DELETE FROM DWHPFX.FILTER_STATUS_TEST WHERE FILTER_STATUS = '{string}'")
connect.import_from_pandas(df[['HOTEL_ID', 'FILTER_STATUS', 'FILTER_LOAD_DATE']], ('DWHPFX', 'FILTER_STATUS_TEST'))

In [38]:
test = {  
   "Records":[  
      {  
         "eventVersion":"2.2",
         "eventSource":"aws:s3",
         "awsRegion":"us-west-2",
         "eventTime":"The time, in ISO-8601 format, for example, 1970-01-01T00:00:00.000Z, when Amazon S3 finished processing the request",
         "eventName":"event-type",
         "userIdentity":{  
            "principalId":"Amazon-customer-ID-of-the-user-who-caused-the-event"
         },
         "requestParameters":{  
            "sourceIPAddress":"ip-address-where-request-came-from"
         },
         "responseElements":{  
            "x-amz-request-id":"Amazon S3 generated request ID",
            "x-amz-id-2":"Amazon S3 host that processed the request"
         },
         "s3":{  
            "s3SchemaVersion":"1.0",
            "configurationId":"ID found in the bucket notification configuration",
            "bucket":{  
               "name":"bucket-name",
               "ownerIdentity":{  
                  "principalId":"Amazon-customer-ID-of-the-bucket-owner"
               },
               "arn":"bucket-ARN"
            },
            "object":{  
               "key":"object-key",
               "size":"object-size",
               "eTag":"object eTag",
               "versionId":"object version if bucket is versioning-enabled, otherwise null",
               "sequencer": "a string representation of a hexadecimal value used to determine event sequence, only used with PUTs and DELETEs"
            }
         },
         "glacierEventData": {
            "restoreEventData": {
               "lifecycleRestorationExpiryTime": "The time, in ISO-8601 format, for example, 1970-01-01T00:00:00.000Z, of Restore Expiry",
               "lifecycleRestoreStorageClass": "Source storage class for restore"
            }
         }
      }
   ]
}

print(test['Records'][0]['s3']['bucket']['name'])

In [40]:
print(test['Records'][0]['s3']['object']['key'])

In [41]:
df

In [45]:
for index, row in df.iterrows():
    print(row['HOTEL_ID'])

In [7]:
import pandas as pd
from datetime import date
import re
### LOCATION OF THE FILE
path = 'C:\\Users\\USER\\Documents\\HPO\\HS_Files\\Archiv\\'
# filename = 'GREYLIST_(2018-07-16).xlsx'
filename = 'BLACKLIST_2019-09-27.xlsx'
string = re.split('_', filename)[0]
df = pd.read_excel(path+filename)
df.rename(columns={'Blacklist or Graylist': 'FILTER_STATUS', 'Hkey' : 'HOTEL_ID'}, errors='raise', inplace=True)
df['FILTER_LOAD_DATE'] = date.today()

# df = df[df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()]
# df['FILTER_STATUS'] = string

In [9]:
df['FILTER_STATUS'].str.upper()==('black' if string == 'BLACKLIST' else 'gray').upper()